# Exp 23: Head-erase test, cats

Card: [exp23_head_erase_test.md](../../experiments/exp23_head_erase_test.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image

from core.augment import FACE_BOXES, read_boxes
from core.box import Box
from core.config import ROOT, TRAIN_DIR
from core.cv import VIP, Config
from core.data import read_train
from core.gradcam import GradCamFigures

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    N_BOXES = 2  # images in box figure
    N_ROWS = 4  # positives in recall test
    N_SAMPLE = 2  # images per Grad-CAM figure
else:
    N_BOXES = 16
    N_ROWS = None  # all generic positives with face box
    N_SAMPLE = 16  # exp19 sample
SEED = 0  # exp19 sample, random boxes
GROW = 2.0  # head box: face box scaled in width and height
CLS = "Cat"
FOLD = 0
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIGS = {
    "exp16": CONFIG_DIR / "exp16_downsampling" / "resnet18.toml",
    "exp20": CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.toml",
    "exp21": CONFIG_DIR / "exp21_face_erase" / "resnet18.toml",
}
CONDITIONS = ["clean", "random box", "head erased"]
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp23_head_erase_test"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "exp20": "#2a78d6", "exp21": "#1baf7a"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

## Head boxes


In [ ]:
faces = read_boxes(FACE_BOXES)
train = read_train()
face = {
    i: dict(faces.get(image_id, [])).get(CLS)
    for i, image_id in train["image_id"].items()
}
generic = train.index[
    (train[CLS] == 1)
    & ~train[VIP].any(axis=1)
    & np.array([face[i] is not None for i in train.index])
]
figures = {}
sample = (
    train.loc[generic].query("fold != @FOLD").sample(N_BOXES, random_state=SEED).index
)
fig, axes = plt.subplots(2, N_BOXES // 2, figsize=(1.1 * N_BOXES, 4.8), squeeze=False)
for ax, i in zip(axes.flat, sample):
    ax.imshow(
        Image.open(TRAIN_DIR / f"{train.loc[i, 'image_id']}.jpg")
        .convert("RGB")
        .resize((224, 224))
    )
    for (x0, y0, x1, y1), color in [
        (face[i], "#eb6834"),
        (Box(*face[i]).grow(GROW), "#2a78d6"),
    ]:
        ax.add_patch(
            Rectangle(
                (x0 * 224, y0 * 224),
                (x1 - x0) * 224,
                (y1 - y0) * 224,
                fill=False,
                color=color,
                lw=2,
            )
        )
for ax in axes.flat:
    ax.set_axis_off()
fig.suptitle(
    f"{CLS} · face box (orange), head box {GROW}x (blue) · fold {FOLD} train part, generic",
    color="#52514e",
)
fig.tight_layout()
figures[f"{PREFIX}_head_boxes.png"] = fig
plt.show()

## Recall: clean, random box, head erased


In [ ]:
cams = {name: GradCamFigures(Config.from_toml(path)) for name, path in CONFIGS.items()}
rng = np.random.default_rng(SEED)
rows = [i for i in generic if train.loc[i, "fold"] in (0, 1)][:N_ROWS]
head = {i: Box(*face[i]).grow(GROW) for i in rows}
boxes = {
    "clean": {},
    "random box": {i: Box(*head[i]).elsewhere(rng) for i in rows},
    "head erased": head,
}
recall = {
    (name, cond): (cam.probs(rows, CLS, boxes[cond]) >= cam.thresholds[CLS]).mean()
    for name, cam in cams.items()
    for cond in CONDITIONS
}
fig, ax = plt.subplots(figsize=(5, 3.8))
for name in cams:
    y = [recall[name, cond] for cond in CONDITIONS]
    ax.plot(CONDITIONS, y, marker="o", lw=2, color=SERIES[name], label=name)
    ax.annotate(
        f"{y[-1]:.3f}",
        (2, y[-1]),
        xytext=(6, 0),
        textcoords="offset points",
        va="center",
        fontsize=8,
        color=SERIES[name],
    )
ax.set(
    title=f"{CLS} ({len(rows)} generic positives)",
    ylabel="recall at own best-F1 threshold",
    xlim=(-0.2, 2.5),
)
ax.legend(loc="lower left", alignment="left")
fig.tight_layout()
figures[f"{PREFIX}_recall.png"] = fig
for key, value in recall.items():
    print(*key, f"{value:.3f}")
plt.show()

## Grad-CAM with head erased


In [ ]:
source = "exp21" if SMOKE else "exp16"  # exp19 rows drawn from exp16 OOF
sample = cams[source].positive_rows(CLS, N_SAMPLE, SEED)
erased = {i: Box(*face[i]).grow(GROW) for i in sample if face[i] is not None}
fig = GradCamFigures.compare(cams, CLS, sample, boxes=erased)
fig.suptitle(
    f"{CLS} · head erased · validation (model held image out) · {len(sample)} labeled positives",
    color="#52514e",
)
figures[f"{PREFIX}_grad_cam_cat.png"] = fig
plt.show()

In [ ]:
if SAVE_FIGURES:
    for fname, fig in figures.items():
        fig.savefig(FIGURES / fname, dpi=120)